# Author New Trial

Write a model in the notebook, validate it, try it locally, package it to `model.py`, and run it through the standard runner.

In [1]:
from __future__ import annotations

import copy
import os
from dataclasses import asdict
from pathlib import Path

from IPython.display import display

import automl
from automl import data, eval, trial, validate
from automl.model import BaseModel
from automl.runner import run_trial

DRY_RUN = True
BASE_SLUG = "notebook_baseline"
RUN_TRIAL = os.getenv("AUTOML_E2E_NOTEBOOKS") == "1"


/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/.venv/lib/python3.13/site-packages/mlflow/pyfunc/utils/data_validation.py:186: UserWarning: Add type hints to the `predict` method to enable data validation and automatic signature inference during model logging. Check https://mlflow.org/docs/latest/model/python_model.html#type-hint-usage-in-pythonmodel for more details.
  color_warning(


In [2]:
active = automl.use_project(dry_run=DRY_RUN)
config = active.config
display(
    {
        "project": active.project_name,
        "repo_root": str(config.repo_root),
        "project_dir": str(config.project_dir),
        "experiment": active.active_experiment_id,
        "dry_run": active.dry_run,
    }
)

loaded = data.materialize(session=active)
run_config = active.config.require_run_config()
train = data.load_dataset(split_name=run_config.train_split, session=active)
holdout = data.load_dataset(split_name=run_config.eval_split, session=active)
loaded.dataset.id


{'project': 'example_homecredit',
 'repo_root': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor',
 'project_dir': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/projects/example_homecredit',
 'experiment': 'example-homecredit',
 'dry_run': True}

'v1_8471c462'

In [3]:
class NotebookModel(BaseModel):
    name = "notebook_model"

    def fit(self, df_train, registry, seed=0):
        from sklearn.compose import ColumnTransformer
        from sklearn.dummy import DummyClassifier
        from sklearn.impute import SimpleImputer
        from sklearn.pipeline import Pipeline

        target = registry.get_by_flag("target")[0]
        required_entries = self.required_transformer_entries()
        required_columns = [
            col for _, _, cols in required_entries for col in cols
        ]
        numeric_columns = [
            col
            for col in registry.get_by_flag("feature")
            if col not in {target, "SPLITID", *required_columns}
            and col in df_train.select_dtypes(include="number").columns
        ]
        self.feature_cols = [*required_columns, *numeric_columns]
        self.preprocessor = ColumnTransformer(
            [
                *required_entries,
                ("numeric", Pipeline([("imputer", SimpleImputer())]), numeric_columns),
            ]
        )
        self.model = DummyClassifier(strategy="prior", random_state=seed)
        X = self.preprocessor.fit_transform(df_train, df_train[target])
        self.model.fit(X, df_train[target])
        self.feature_registry = copy.deepcopy(registry)
        self.feature_registry.set_flag(self.feature_registry.get_by_flag("feature"), "model", False)
        for col in self.feature_cols:
            self.feature_registry.set_flag(col, "model", True)

    def transform(self, df):
        return self.preprocessor.transform(df)

    def _predict(self, X):
        return self.model.predict_proba(X)[:, 1]


In [4]:
validate.model(NotebookModel, df=train.df, registry=train.registry, session=active)


ValidationReport(issues=[], schema_version=1)

In [5]:
model = NotebookModel()
model.fit(train.df, train.registry)
target = train.registry.get_by_flag("target")[0]
pred = model.predict(model_input=holdout.df.drop(columns=[target]))
eval.evaluate_frame(
    y_pred=pred,
    df=holdout.df,
    spec=active.config.require_eval_spec(),
    target_col=target,
    session=active,
)


{'primary': 'auc',
 'metrics': [{'name': 'auc', 'value': 0.5, 'augmentations': []}]}

In [6]:
model_path = trial.package_model(
    NotebookModel,
    imports=["import copy", "from automl.model import BaseModel"],
    output_path=Path("scratch/notebook_model.py"),
)
model_path


PosixPath('scratch/notebook_model.py')

In [ ]:
# Manual Hack 
# RUN_TRIAL = True

In [9]:
draft_dir = None
draft_slug = None
slug_index = 1
while draft_dir is None:
    candidate_slug = BASE_SLUG if slug_index == 1 else f"{BASE_SLUG}_{slug_index}"
    try:
        draft_dir = trial.create(
            candidate_slug,
            "human_baseline",
            hypothesis="Notebook-authored baseline.",
            training_origin="human",
            model_source=model_path,
            session=active,
        )
        draft_slug = candidate_slug
    except FileExistsError:
        slug_index += 1

run_result = None
if RUN_TRIAL:
    run_result = run_trial(draft_dir, session=active)
    if run_result.status != "FINISHED":
        raise RuntimeError(run_result.error or f"trial run failed: {run_result.status}")

{
    "draft_slug": draft_slug,
    "draft_dir": str(draft_dir),
    "model_path": str(draft_dir / "model.py"),
    "run_result": asdict(run_result) if run_result is not None else None,
    "run_command": f"uv run automl --project {active.project_name} trial run {draft_dir}",
}


2026/06/02 00:07:58 INFO mlflow.pyfunc: Validating input example against model signature
2026/06/02 00:08:00 WARNING mlflow.utils.requirements_utils: The following packages were not found in the public PyPI package index as of 2025-04-15; if these packages are not present in the public PyPI index, you must install them manually before loading your model: {'brigit-automl'}
2026/06/02 00:08:00 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run notebook_baseline_2 at: http://127.0.0.1:54321/#/experiments/2/runs/b8b52c8012ea47638beee888781c37bf
🧪 View experiment at: http://127.0.0.1:54321/#/experiments/2
🏃 View run notebook_baseline_2 at: http://127.0.0.1:54321/#/experiments/2/runs/b8b52c8012ea47638beee888781c37bf
🧪 View experiment at: http://127.0.0.1:54321/#/experiments/2


{'draft_slug': 'notebook_baseline_2',
 'draft_dir': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/projects/example_homecredit/experiments/dry_run/example_homecredit/example-homecredit/notebook_baseline_2',
 'model_path': '/Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/projects/example_homecredit/experiments/dry_run/example_homecredit/example-homecredit/notebook_baseline_2/model.py',
 'run_result': {'status': 'FINISHED',
  'run_id': 'b8b52c8012ea47638beee888781c37bf',
  'trial_id': '2_notebook_baseline_2',
  'trial_number': 2,
  'metrics': {'auc': 0.5},
  'error': None},
 'run_command': 'uv run automl --project example_homecredit trial run /Users/zhengisamazing/1.python_dir/brigit/automl_dev-refactor/projects/example_homecredit/experiments/dry_run/example_homecredit/example-homecredit/notebook_baseline_2'}